# S1_01_summary_retry_dataset

This notebook runs a function acts as a recovery auditor. It identifies exactly which records failed to process and packages them into a new dataset for the next retry phase (Month 13/14).

Core Workflow
Auditing: Compares the source list (original .npz) against the success list (sharded .jsonl outputs) to find missing Job IDs.

Extraction: Filters the original .npz data to keep only the rows that correspond to those missing IDs.

Packaging: Saves two specific files into the month's directory:

_missing_job_ids.txt: A plain list of IDs that failed.

_retry.npz: A compressed data file containing the full records for those failures.

Reporting: Outputs a statistical summary (expected vs. produced) and a few random samples of the successful summaries for manual inspection.

# Recovery function

In [5]:
from pathlib import Path
import glob, json, random
import numpy as np

def month_missing_ids_and_retry_npz(
    MONTH: str,
    root="/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months",
    id_keys=("id", "job_id", "job_ids", "ids", "adzuna_id"),
    sample_n=3,
):
    DIR = Path(root) / f"{MONTH}_npz"
    NPZ_PATH = DIR / f"{MONTH}.npz"

    MISSING_TXT = DIR / f"{MONTH}_missing_job_ids.txt"
    RETRY_NPZ   = DIR / f"{MONTH}_retry.npz"

    if not NPZ_PATH.exists():
        raise FileNotFoundError(f"Missing NPZ: {NPZ_PATH}")

    # ----------------------------
    # 1) Load NPZ + expected IDs
    # ----------------------------
    npz = np.load(NPZ_PATH, allow_pickle=True)
    key = next((k for k in id_keys if k in npz.files), None)
    if key is None:
        raise KeyError(f"No id key found in NPZ. Have: {npz.files}")

    ids_npz = np.asarray(npz[key]).astype(str)
    n_expected = ids_npz.size

    # For missing detection we only need uniques (fast + memory sane)
    ids_unique = np.unique(ids_npz)
    ids_unique.sort()

    # ----------------------------
    # 2) Scan JSONL shards, collect produced IDs
    #    Fast path: avoid keeping full parsed objects, just extract id field.
    # ----------------------------
    files = sorted(glob.glob(str(DIR / f"{MONTH}_extract_*.jsonl")))
    if not files:
        raise FileNotFoundError(f"No JSONL outputs found under: {DIR}")

    produced = set()
    bad_json = 0
    total_lines = 0

    # small sampling without extra passes
    sample_rows = []
    sample_target_file = random.choice(files)

    for fn in files:
        with open(fn, "r", encoding="utf-8") as f:
            for line in f:
                total_lines += 1
                try:
                    obj = json.loads(line)
                except json.JSONDecodeError:
                    bad_json += 1
                    continue

                jid = obj.get("id") or obj.get("job_id") or obj.get("adzuna_id")
                if jid is not None:
                    produced.add(str(jid))

                if fn == sample_target_file and len(sample_rows) < sample_n:
                    sample_rows.append(obj)

    produced_arr = np.fromiter(produced, dtype=object, count=len(produced))
    produced_arr.sort()

    # ----------------------------
    # 3) Missing = ids_unique \ produced_arr  (safe searchsorted)
    # ----------------------------
    idx = np.searchsorted(produced_arr, ids_unique)

    in_produced = np.zeros(ids_unique.shape[0], dtype=bool)
    ok = idx < produced_arr.size
    in_produced[ok] = (produced_arr[idx[ok]] == ids_unique[ok])

    missing_arr = ids_unique[~in_produced]

    # write missing txt
    MISSING_TXT.write_text(
        "\n".join(missing_arr.tolist()) + ("\n" if missing_arr.size else ""),
        encoding="utf-8",
    )

    # ----------------------------
    # 4) Build retry NPZ (preserve original row order + duplicates)
    #    Make mask via searchsorted into missing_arr (also safe)
    # ----------------------------
    missing_sorted = missing_arr.copy()
    missing_sorted.sort()

    pos = np.searchsorted(missing_sorted, ids_npz)
    mask = np.zeros(ids_npz.shape[0], dtype=bool)
    ok2 = pos < missing_sorted.size
    mask[ok2] = (missing_sorted[pos[ok2]] == ids_npz[ok2])

    out = {k: npz[k][mask] for k in npz.files}
    np.savez_compressed(RETRY_NPZ, **out)

    # ----------------------------
    # 5) Quick stats + sample rows
    # ----------------------------
    report = {
        "month": MONTH,
        "npz_path": str(NPZ_PATH),
        "jsonl_dir": str(DIR),
        "jsonl_files": len(files),
        "expected_rows": int(n_expected),
        "expected_unique_ids": int(ids_unique.size),
        "total_jsonl_lines": int(total_lines),
        "bad_json_lines": int(bad_json),
        "produced_unique_ids": int(produced_arr.size),
        "missing_unique_ids": int(missing_arr.size),
        "missing_txt": str(MISSING_TXT),
        "retry_npz": str(RETRY_NPZ),
        "retry_rows": int(mask.sum()),
    }

    print(report)

    if sample_rows:
        print("\nSAMPLE OUTPUT ROWS (from one random jsonl file):")
        for i, obj in enumerate(sample_rows[:sample_n], 1):
            jid = obj.get("id") or obj.get("job_id") or obj.get("adzuna_id")
            print(f"\n[{i}] id: {jid}")
            jd = (obj.get("job_description") or "")
            print("job_description:", jd[:300])
            print("parsed:", obj.get("parsed"))
            print("error:", obj.get("error"))

    return report

# After run S1_00_demo_LLM_summary_run_across_months
- Retry from Sbatch Loop Months 9, 10, 11 and 12

In [ ]:
MONTH = "adzuna_month09"
#month_missing_ids_and_retry_npz(MONTH)

In [2]:
MONTH = "adzuna_month10"
#month_missing_ids_and_retry_npz(MONTH)

{'month': 'adzuna_month10', 'npz_path': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month10_npz/adzuna_month10.npz', 'jsonl_dir': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month10_npz', 'jsonl_files': 100, 'expected_rows': 2333824, 'expected_unique_ids': 2333824, 'total_jsonl_lines': 2180380, 'bad_json_lines': 0, 'produced_unique_ids': 2180380, 'missing_unique_ids': 153444, 'missing_txt': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month10_npz/adzuna_month10_missing_job_ids.txt', 'retry_npz': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month10_npz/adzuna_month10_retry.npz', 'retry_rows': 153444}

SAMPLE OUTPUT ROWS (from one random jsonl file):

[1] id: 3603284211
job_description: Your main aim will be to carry o

{'month': 'adzuna_month10',
 'npz_path': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month10_npz/adzuna_month10.npz',
 'jsonl_dir': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month10_npz',
 'jsonl_files': 100,
 'expected_rows': 2333824,
 'expected_unique_ids': 2333824,
 'total_jsonl_lines': 2180380,
 'bad_json_lines': 0,
 'produced_unique_ids': 2180380,
 'missing_unique_ids': 153444,
 'missing_txt': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month10_npz/adzuna_month10_missing_job_ids.txt',
 'retry_npz': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month10_npz/adzuna_month10_retry.npz',
 'retry_rows': 153444}

In [3]:
MONTH = "adzuna_month11"
#month_missing_ids_and_retry_npz(MONTH)

{'month': 'adzuna_month11', 'npz_path': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month11_npz/adzuna_month11.npz', 'jsonl_dir': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month11_npz', 'jsonl_files': 100, 'expected_rows': 2192908, 'expected_unique_ids': 2192908, 'total_jsonl_lines': 2041412, 'bad_json_lines': 0, 'produced_unique_ids': 2041412, 'missing_unique_ids': 151496, 'missing_txt': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month11_npz/adzuna_month11_missing_job_ids.txt', 'retry_npz': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month11_npz/adzuna_month11_retry.npz', 'retry_rows': 151496}

SAMPLE OUTPUT ROWS (from one random jsonl file):

[1] id: 3668236268
job_description: Trust Housing Association has a 

{'month': 'adzuna_month11',
 'npz_path': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month11_npz/adzuna_month11.npz',
 'jsonl_dir': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month11_npz',
 'jsonl_files': 100,
 'expected_rows': 2192908,
 'expected_unique_ids': 2192908,
 'total_jsonl_lines': 2041412,
 'bad_json_lines': 0,
 'produced_unique_ids': 2041412,
 'missing_unique_ids': 151496,
 'missing_txt': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month11_npz/adzuna_month11_missing_job_ids.txt',
 'retry_npz': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month11_npz/adzuna_month11_retry.npz',
 'retry_rows': 151496}

In [4]:
MONTH = "adzuna_month12"
#month_missing_ids_and_retry_npz(MONTH)

{'month': 'adzuna_month12', 'npz_path': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month12_npz/adzuna_month12.npz', 'jsonl_dir': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month12_npz', 'jsonl_files': 100, 'expected_rows': 1838220, 'expected_unique_ids': 1838220, 'total_jsonl_lines': 1704280, 'bad_json_lines': 0, 'produced_unique_ids': 1704280, 'missing_unique_ids': 133940, 'missing_txt': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month12_npz/adzuna_month12_missing_job_ids.txt', 'retry_npz': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month12_npz/adzuna_month12_retry.npz', 'retry_rows': 133940}

SAMPLE OUTPUT ROWS (from one random jsonl file):

[1] id: 3622352213
job_description: ASSOCIATE DENTIST, LONDON We're 

{'month': 'adzuna_month12',
 'npz_path': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month12_npz/adzuna_month12.npz',
 'jsonl_dir': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month12_npz',
 'jsonl_files': 100,
 'expected_rows': 1838220,
 'expected_unique_ids': 1838220,
 'total_jsonl_lines': 1704280,
 'bad_json_lines': 0,
 'produced_unique_ids': 1704280,
 'missing_unique_ids': 133940,
 'missing_txt': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month12_npz/adzuna_month12_missing_job_ids.txt',
 'retry_npz': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month12_npz/adzuna_month12_retry.npz',
 'retry_rows': 133940}

# After run S1_00_demo_LLM_summary_run_across_months
- Retry from Sbatch Loop Months 7,8

In [2]:
MONTH = "adzuna_month07"
#month_missing_ids_and_retry_npz(MONTH)

{'month': 'adzuna_month07', 'npz_path': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month07_npz/adzuna_month07.npz', 'jsonl_dir': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month07_npz', 'jsonl_files': 100, 'expected_rows': 2505414, 'expected_unique_ids': 2505414, 'total_jsonl_lines': 2184408, 'bad_json_lines': 0, 'produced_unique_ids': 2184408, 'missing_unique_ids': 321006, 'missing_txt': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month07_npz/adzuna_month07_missing_job_ids.txt', 'retry_npz': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month07_npz/adzuna_month07_retry.npz', 'retry_rows': 321006}

SAMPLE OUTPUT ROWS (from one random jsonl file):

[1] id: 3226715090
job_description: Stafforce are working in partner

{'month': 'adzuna_month07',
 'npz_path': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month07_npz/adzuna_month07.npz',
 'jsonl_dir': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month07_npz',
 'jsonl_files': 100,
 'expected_rows': 2505414,
 'expected_unique_ids': 2505414,
 'total_jsonl_lines': 2184408,
 'bad_json_lines': 0,
 'produced_unique_ids': 2184408,
 'missing_unique_ids': 321006,
 'missing_txt': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month07_npz/adzuna_month07_missing_job_ids.txt',
 'retry_npz': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month07_npz/adzuna_month07_retry.npz',
 'retry_rows': 321006}

In [3]:
MONTH = "adzuna_month08"
#month_missing_ids_and_retry_npz(MONTH)

{'month': 'adzuna_month08', 'npz_path': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month08_npz/adzuna_month08.npz', 'jsonl_dir': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month08_npz', 'jsonl_files': 100, 'expected_rows': 2164579, 'expected_unique_ids': 2164579, 'total_jsonl_lines': 1914855, 'bad_json_lines': 0, 'produced_unique_ids': 1914855, 'missing_unique_ids': 249724, 'missing_txt': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month08_npz/adzuna_month08_missing_job_ids.txt', 'retry_npz': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month08_npz/adzuna_month08_retry.npz', 'retry_rows': 249724}

SAMPLE OUTPUT ROWS (from one random jsonl file):

[1] id: 3340406651
job_description: Full Time / Flexible shift patte

{'month': 'adzuna_month08',
 'npz_path': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month08_npz/adzuna_month08.npz',
 'jsonl_dir': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month08_npz',
 'jsonl_files': 100,
 'expected_rows': 2164579,
 'expected_unique_ids': 2164579,
 'total_jsonl_lines': 1914855,
 'bad_json_lines': 0,
 'produced_unique_ids': 1914855,
 'missing_unique_ids': 249724,
 'missing_txt': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month08_npz/adzuna_month08_missing_job_ids.txt',
 'retry_npz': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month08_npz/adzuna_month08_retry.npz',
 'retry_rows': 249724}

# After run S1_00_demo_LLM_summary_run_across_months
- Retry from Sbatch Loop Months 1 to 4

In [5]:
MONTH = "adzuna_month01"
#month_missing_ids_and_retry_npz(MONTH)

{'month': 'adzuna_month01', 'npz_path': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month01_npz/adzuna_month01.npz', 'jsonl_dir': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month01_npz', 'jsonl_files': 100, 'expected_rows': 2840180, 'expected_unique_ids': 2840180, 'total_jsonl_lines': 2658928, 'bad_json_lines': 0, 'produced_unique_ids': 2658928, 'missing_unique_ids': 181252, 'missing_txt': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month01_npz/adzuna_month01_missing_job_ids.txt', 'retry_npz': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month01_npz/adzuna_month01_retry.npz', 'retry_rows': 181252}

SAMPLE OUTPUT ROWS (from one random jsonl file):

[1] id: 2780654438
job_description: Role Description: UK Financial S

{'month': 'adzuna_month01',
 'npz_path': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month01_npz/adzuna_month01.npz',
 'jsonl_dir': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month01_npz',
 'jsonl_files': 100,
 'expected_rows': 2840180,
 'expected_unique_ids': 2840180,
 'total_jsonl_lines': 2658928,
 'bad_json_lines': 0,
 'produced_unique_ids': 2658928,
 'missing_unique_ids': 181252,
 'missing_txt': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month01_npz/adzuna_month01_missing_job_ids.txt',
 'retry_npz': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month01_npz/adzuna_month01_retry.npz',
 'retry_rows': 181252}

In [6]:
MONTH = "adzuna_month02"
#month_missing_ids_and_retry_npz(MONTH)

{'month': 'adzuna_month02', 'npz_path': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month02_npz/adzuna_month02.npz', 'jsonl_dir': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month02_npz', 'jsonl_files': 100, 'expected_rows': 2692671, 'expected_unique_ids': 2692671, 'total_jsonl_lines': 2230469, 'bad_json_lines': 0, 'produced_unique_ids': 2230469, 'missing_unique_ids': 462202, 'missing_txt': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month02_npz/adzuna_month02_missing_job_ids.txt', 'retry_npz': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month02_npz/adzuna_month02_retry.npz', 'retry_rows': 462202}

SAMPLE OUTPUT ROWS (from one random jsonl file):

[1] id: 2909336960
job_description: Waiting Staff As Waiting Staff, 

{'month': 'adzuna_month02',
 'npz_path': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month02_npz/adzuna_month02.npz',
 'jsonl_dir': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month02_npz',
 'jsonl_files': 100,
 'expected_rows': 2692671,
 'expected_unique_ids': 2692671,
 'total_jsonl_lines': 2230469,
 'bad_json_lines': 0,
 'produced_unique_ids': 2230469,
 'missing_unique_ids': 462202,
 'missing_txt': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month02_npz/adzuna_month02_missing_job_ids.txt',
 'retry_npz': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month02_npz/adzuna_month02_retry.npz',
 'retry_rows': 462202}

In [7]:
MONTH = "adzuna_month03"
#month_missing_ids_and_retry_npz(MONTH)

{'month': 'adzuna_month03', 'npz_path': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month03_npz/adzuna_month03.npz', 'jsonl_dir': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month03_npz', 'jsonl_files': 100, 'expected_rows': 2521202, 'expected_unique_ids': 2521202, 'total_jsonl_lines': 2304918, 'bad_json_lines': 0, 'produced_unique_ids': 2304918, 'missing_unique_ids': 216284, 'missing_txt': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month03_npz/adzuna_month03_missing_job_ids.txt', 'retry_npz': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month03_npz/adzuna_month03_retry.npz', 'retry_rows': 216284}

SAMPLE OUTPUT ROWS (from one random jsonl file):

[1] id: 2970275332
job_description: Our client is a well-established

{'month': 'adzuna_month03',
 'npz_path': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month03_npz/adzuna_month03.npz',
 'jsonl_dir': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month03_npz',
 'jsonl_files': 100,
 'expected_rows': 2521202,
 'expected_unique_ids': 2521202,
 'total_jsonl_lines': 2304918,
 'bad_json_lines': 0,
 'produced_unique_ids': 2304918,
 'missing_unique_ids': 216284,
 'missing_txt': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month03_npz/adzuna_month03_missing_job_ids.txt',
 'retry_npz': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month03_npz/adzuna_month03_retry.npz',
 'retry_rows': 216284}

In [8]:
MONTH = "adzuna_month04"
#month_missing_ids_and_retry_npz(MONTH)

{'month': 'adzuna_month04', 'npz_path': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month04_npz/adzuna_month04.npz', 'jsonl_dir': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month04_npz', 'jsonl_files': 100, 'expected_rows': 2492751, 'expected_unique_ids': 2492751, 'total_jsonl_lines': 2326927, 'bad_json_lines': 0, 'produced_unique_ids': 2326927, 'missing_unique_ids': 165824, 'missing_txt': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month04_npz/adzuna_month04_missing_job_ids.txt', 'retry_npz': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month04_npz/adzuna_month04_retry.npz', 'retry_rows': 165824}

SAMPLE OUTPUT ROWS (from one random jsonl file):

[1] id: 3073517152
job_description: Castle View Group have an exciti

{'month': 'adzuna_month04',
 'npz_path': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month04_npz/adzuna_month04.npz',
 'jsonl_dir': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month04_npz',
 'jsonl_files': 100,
 'expected_rows': 2492751,
 'expected_unique_ids': 2492751,
 'total_jsonl_lines': 2326927,
 'bad_json_lines': 0,
 'produced_unique_ids': 2326927,
 'missing_unique_ids': 165824,
 'missing_txt': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month04_npz/adzuna_month04_missing_job_ids.txt',
 'retry_npz': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month04_npz/adzuna_month04_retry.npz',
 'retry_rows': 165824}

# After run S1_00_demo_LLM_summary_run_across_months
- Retry from Sbatch Loop Months 5 to 6

In [9]:
MONTH = "adzuna_month05"
#month_missing_ids_and_retry_npz(MONTH)

{'month': 'adzuna_month05', 'npz_path': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month05_npz/adzuna_month05.npz', 'jsonl_dir': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month05_npz', 'jsonl_files': 100, 'expected_rows': 2684650, 'expected_unique_ids': 2684650, 'total_jsonl_lines': 2547671, 'bad_json_lines': 0, 'produced_unique_ids': 2547671, 'missing_unique_ids': 136979, 'missing_txt': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month05_npz/adzuna_month05_missing_job_ids.txt', 'retry_npz': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month05_npz/adzuna_month05_retry.npz', 'retry_rows': 136979}

SAMPLE OUTPUT ROWS (from one random jsonl file):

[1] id: 3084378929
job_description: The Job: Night Manager required 

{'month': 'adzuna_month05',
 'npz_path': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month05_npz/adzuna_month05.npz',
 'jsonl_dir': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month05_npz',
 'jsonl_files': 100,
 'expected_rows': 2684650,
 'expected_unique_ids': 2684650,
 'total_jsonl_lines': 2547671,
 'bad_json_lines': 0,
 'produced_unique_ids': 2547671,
 'missing_unique_ids': 136979,
 'missing_txt': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month05_npz/adzuna_month05_missing_job_ids.txt',
 'retry_npz': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month05_npz/adzuna_month05_retry.npz',
 'retry_rows': 136979}

In [2]:
MONTH = "adzuna_month06"
#month_missing_ids_and_retry_npz(MONTH)

{'month': 'adzuna_month06', 'npz_path': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month06_npz/adzuna_month06.npz', 'jsonl_dir': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month06_npz', 'jsonl_files': 100, 'expected_rows': 2280484, 'expected_unique_ids': 2280484, 'total_jsonl_lines': 1993750, 'bad_json_lines': 0, 'produced_unique_ids': 1993750, 'missing_unique_ids': 286734, 'missing_txt': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month06_npz/adzuna_month06_missing_job_ids.txt', 'retry_npz': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month06_npz/adzuna_month06_retry.npz', 'retry_rows': 286734}

SAMPLE OUTPUT ROWS (from one random jsonl file):

[1] id: 3237388469
job_description: Workshop Fitter - Powered Access

{'month': 'adzuna_month06',
 'npz_path': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month06_npz/adzuna_month06.npz',
 'jsonl_dir': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month06_npz',
 'jsonl_files': 100,
 'expected_rows': 2280484,
 'expected_unique_ids': 2280484,
 'total_jsonl_lines': 1993750,
 'bad_json_lines': 0,
 'produced_unique_ids': 1993750,
 'missing_unique_ids': 286734,
 'missing_txt': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month06_npz/adzuna_month06_missing_job_ids.txt',
 'retry_npz': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month06_npz/adzuna_month06_retry.npz',
 'retry_rows': 286734}

Put here code that consolidatedd all missings in previous runs into `adzuna_month13`

# Retries retrieving the 10% missing ones

in order to keep in line of format we 1st retry the single dataset derived see ùntitled3.ipynb the one built from each month suffix "_retry""

In [6]:
MONTH = "adzuna_month13"
#month_missing_ids_and_retry_npz(MONTH)

{'month': 'adzuna_month13', 'npz_path': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month13_npz/adzuna_month13.npz', 'jsonl_dir': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month13_npz', 'jsonl_files': 100, 'expected_rows': 2573657, 'expected_unique_ids': 2573657, 'total_jsonl_lines': 2551368, 'bad_json_lines': 0, 'produced_unique_ids': 2551368, 'missing_unique_ids': 22289, 'missing_txt': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month13_npz/adzuna_month13_missing_job_ids.txt', 'retry_npz': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month13_npz/adzuna_month13_retry.npz', 'retry_rows': 22289}

SAMPLE OUTPUT ROWS (from one random jsonl file):

[1] id: 3365407804
job_description: 
parsed: {'short_description': 'Le

{'month': 'adzuna_month13',
 'npz_path': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month13_npz/adzuna_month13.npz',
 'jsonl_dir': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month13_npz',
 'jsonl_files': 100,
 'expected_rows': 2573657,
 'expected_unique_ids': 2573657,
 'total_jsonl_lines': 2551368,
 'bad_json_lines': 0,
 'produced_unique_ids': 2551368,
 'missing_unique_ids': 22289,
 'missing_txt': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month13_npz/adzuna_month13_missing_job_ids.txt',
 'retry_npz': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month13_npz/adzuna_month13_retry.npz',
 'retry_rows': 22289}

create next retry dataset `adzuna_month14`

In [7]:
from pathlib import Path
import shutil
import numpy as np

src = Path("/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month13_npz/adzuna_month13_retry.npz")

root = src.parents[1]
month_folder = "adzuna_month14_npz"

out_dir = root / month_folder
out_dir.mkdir(parents=True, exist_ok=True)

dst = out_dir / (month_folder.replace("_npz", "") + ".npz")

# copy (non-destructive)
shutil.copy2(src, dst)

# patch: description = job_description
z = np.load(dst, allow_pickle=True)
out = dict(z)
z.close()

if "description" not in out:
    out["description"] = out.get("job_description", np.array([], dtype=object))

np.savez(dst, **out)

print("written and patched:", dst)
print("keys:", sorted(out.keys()))


written and patched: /projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month14_npz/adzuna_month14.npz
keys: ['category_name', 'description', 'id', 'job_description', 'source_month', 'title']


In [8]:
MONTH = "adzuna_month14"
month_missing_ids_and_retry_npz(MONTH)

{'month': 'adzuna_month14', 'npz_path': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month14_npz/adzuna_month14.npz', 'jsonl_dir': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month14_npz', 'jsonl_files': 100, 'expected_rows': 22289, 'expected_unique_ids': 22289, 'total_jsonl_lines': 22039, 'bad_json_lines': 0, 'produced_unique_ids': 22039, 'missing_unique_ids': 250, 'missing_txt': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month14_npz/adzuna_month14_missing_job_ids.txt', 'retry_npz': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month14_npz/adzuna_month14_retry.npz', 'retry_rows': 250}

SAMPLE OUTPUT ROWS (from one random jsonl file):

[1] id: 3146353271
job_description: 
parsed: {'short_description': 'We are seeking

{'month': 'adzuna_month14',
 'npz_path': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month14_npz/adzuna_month14.npz',
 'jsonl_dir': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month14_npz',
 'jsonl_files': 100,
 'expected_rows': 22289,
 'expected_unique_ids': 22289,
 'total_jsonl_lines': 22039,
 'bad_json_lines': 0,
 'produced_unique_ids': 22039,
 'missing_unique_ids': 250,
 'missing_txt': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month14_npz/adzuna_month14_missing_job_ids.txt',
 'retry_npz': '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month14_npz/adzuna_month14_retry.npz',
 'retry_rows': 250}

In [ ]:
# Now lets move it to AISI_demo folder

In [1]:
import shutil
from pathlib import Path

# Setup paths
src_root = Path("/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months")
dst_root = Path("/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_1_LLM_summary")

# Ensure the destination base directory exists
dst_root.mkdir(parents=True, exist_ok=True)

# Loop through and copy each folder
for folder in src_root.glob("adzuna_month*_npz"):
    dest_path = dst_root / folder.name
    
    if not dest_path.exists():
        shutil.copytree(folder, dest_path)
        print(f"Successfully copied: {folder.name}")
    else:
        print(f"Skipped: {folder.name} (already exists at destination)")

Successfully copied: adzuna_month12_npz
Successfully copied: adzuna_month05_npz
Successfully copied: adzuna_month03_npz
Successfully copied: adzuna_month14_npz
Successfully copied: adzuna_month10_npz
Successfully copied: adzuna_month06_npz
Successfully copied: adzuna_month02_npz
Successfully copied: adzuna_month01_npz
Successfully copied: adzuna_month09_npz
Successfully copied: adzuna_month07_npz
Successfully copied: adzuna_month04_npz
Successfully copied: adzuna_month11_npz
Successfully copied: adzuna_month13_npz
Successfully copied: adzuna_month08_npz
